# carGO PH — Blockchain Data Retrieval
**Course:** MO-IT148 — Application Development and Emerging Technologies <br>
**Group:** NodeBlk <br><br><br>
**Week:** 6 — Data Retrieval + Cleaning + Stats <br>
**Description:** Retrieves IoT sensor records (GPS, Temperature, RFID) from the IoTDataStorage smart contract on Ganache via Web3.py. Cleans and structures the data into a unified DataFrame for statistical analysis and export.<br>


> ⚠️ Prerequisite: Ganache must be running with the contract deployed before executing the Setup cell.

## 1. Setup
Imports, ABI load, and contract instantiation. Ganache must be running before this cell executes. All downstream cells depend on `logistics_contract`.

In [10]:
# ── Setup ──────────────────────────────────────────────────────────────────
# Imports, ABI load, contract instantiation, connection confirmation

from web3 import Web3
import pandas as pd
import numpy as np
import json

# Load ABI from compiled contract data
with open('../contracts/IoTDataStorage_compData.json') as f:
    comp_data = json.load(f)

abi = json.loads(comp_data['metadata'])['output']['abi']
# comp_data['metadata'] is a JSON string inside the JSON file — needs a second parse

# Contract address — update this each new Ganache session
CONTRACT_ADDRESS = Web3.to_checksum_address("0x2728C1070A6a3E4c42A82537Db99AA828F3E01aA")

# Connect to Ganache
web3 = Web3(Web3.HTTPProvider("http://127.0.0.1:7545"))
logistics_contract = web3.eth.contract(address=CONTRACT_ADDRESS, abi=abi)
web3.eth.default_account = web3.eth.accounts[0]

# Confirm connection
print("Connected:", web3.is_connected())
print("GPS records on chain:", logistics_contract.functions.gpsRecordCount().call())
print("Temp records on chain:", logistics_contract.functions.tempRecordCount().call())
print("RFID records on chain:", logistics_contract.functions.rfidRecordCount().call())

Connected: True
GPS records on chain: 150
Temp records on chain: 95
RFID records on chain: 84


## 2. Retrieval
Loops through GPS, Temperature, and RFID records stored on-chain. Collects raw records into lists for DataFrame construction.

In [11]:
# ── Retrieval ───────────────────────────────────────────────────────────────

# GPS records
gps_count = logistics_contract.functions.gpsRecordCount().call()
gps_rows = []
for i in range(gps_count):
    r = logistics_contract.functions.gpsRecords(i).call()
    gps_rows.append({
        "timestamp": r[0], "rfid_tag": r[1], "device_id": r[2],
        "latitude": r[3], "longitude": r[4], "sensor_type": "GPS"
    })
gps_records_df = pd.DataFrame(gps_rows)
if gps_records_df.empty:
    print("⚠️ Warning: No GPS records retrieved. Check Ganache session.")

# Temperature records
temp_count = logistics_contract.functions.tempRecordCount().call()
temp_rows = []
for i in range(temp_count):
    r = logistics_contract.functions.tempRecords(i).call()
    temp_rows.append({
        "timestamp": r[0], "rfid_tag": r[1], "device_id": r[2],
        "temperature_raw": r[3], "sensor_type": "Temperature"
    })
temp_records_df = pd.DataFrame(temp_rows)
if temp_records_df.empty:
    print("⚠️ Warning: No Temperature records retrieved. Check Ganache session.")

# RFID records
rfid_count = logistics_contract.functions.rfidRecordCount().call()
rfid_rows = []
for i in range(rfid_count):
    r = logistics_contract.functions.rfidRecords(i).call()
    rfid_rows.append({
        "timestamp": r[0], "rfid_tag": r[1], "device_id": r[2],
        "scan_status": r[3], "sensor_type": "RFID"
    })
rfid_records_df = pd.DataFrame(rfid_rows)
if rfid_records_df.empty:
    print("⚠️ Warning: No RFID records retrieved. Check Ganache session.")

## 3. DataFrame Construction
Converts raw record lists into structured DataFrames — one per sensor type.

In [12]:
# ── DataFrame Construction ──────────────────────────────────────────────────

iot_records_df = pd.concat(
    [gps_records_df, temp_records_df, rfid_records_df],
    ignore_index=True
)
# ignore_index=True — resets to clean 0-based index after stacking three DataFrames
# columns absent for a sensor type (e.g. latitude for Temperature rows) become NaN

print(f"Total records retrieved: {len(iot_records_df)}")
print(iot_records_df.dtypes)

Total records retrieved: 329
timestamp            int64
rfid_tag            object
device_id           object
latitude            object
longitude           object
sensor_type         object
temperature_raw    float64
scan_status         object
dtype: object


## 4. Cleaning
Type conversions and data quality. Unix timestamps → datetime, tempTimes10 ÷ 10 → °C, drop NaN, reset index.

In [13]:
# ── Cleaning ────────────────────────────────────────────────────────────────

iot_cleaned_df = iot_records_df.copy()
# copy() — keeps iot_records_df intact; avoids SettingWithCopyWarning

# Rename blockchain timestamp (differentiate it to sensor timestamp))
iot_cleaned_df.rename(columns={'timestamp': 'blockchain_timestamp'}, inplace=True)

# blockchain_timestamp — Unix int → datetime
iot_cleaned_df['blockchain_timestamp'] = pd.to_datetime(
    iot_cleaned_df['blockchain_timestamp'], unit='s'
)

# Latitude/longitude — string → float
iot_cleaned_df['latitude'] = pd.to_numeric(iot_cleaned_df['latitude'], errors='coerce')
iot_cleaned_df['longitude'] = pd.to_numeric(iot_cleaned_df['longitude'], errors='coerce')

# Temperature — int × 10 → float °C
iot_cleaned_df['temperature_c'] = iot_cleaned_df['temperature_raw'] / 10
iot_cleaned_df.drop(columns=['temperature_raw'], inplace=True)

# Drop NaN on identity fields only
iot_cleaned_df.dropna(subset=['rfid_tag', 'sensor_type'], inplace=True)
iot_cleaned_df.reset_index(drop=True, inplace=True)

# ── Sensor Timestamp Merge (the ONE csv pull) ────────────────────────────────
# sensor_timestamp was never stored on-chain — only block.timestamp was recorded.
# iot_data.csv holds the original simulation timestamps.
# Join key: rfid_tag + device_id (unique per reading across both files)

iot_source_df = pd.read_csv('../data/iot_data.csv')
iot_source_df = iot_source_df[['rfid_tag', 'device_id', 'timestamp']].rename(
    columns={'timestamp': 'sensor_timestamp'}
)
iot_source_df['sensor_timestamp'] = pd.to_datetime(iot_source_df['sensor_timestamp'])

iot_source_df = iot_source_df.drop_duplicates(subset=['rfid_tag', 'device_id'])

iot_cleaned_df = iot_cleaned_df.merge(
    iot_source_df,
    on=['rfid_tag', 'device_id'],
    how='left'
)

# Exclude the TEST-001 dummy immediately after merge
iot_cleaned_df = iot_cleaned_df[iot_cleaned_df['rfid_tag'] != 'TEST-001']

# Reorder columns so sensor_timestamp follows blockchain_timestamp
iot_cleaned_df = iot_cleaned_df[
    ['blockchain_timestamp', 'sensor_timestamp'] +
    [c for c in iot_cleaned_df.columns if c not in ['blockchain_timestamp','sensor_timestamp']]
]

# Rows without a match (e.g. the TEST-001 dummy) will have NaT for sensor_timestamp — expected

# ── is_flagged boolean ───────────────────────────────────────────────────────
# Derived directly from scan_status — no CSV needed
iot_cleaned_df['is_flagged'] = iot_cleaned_df['scan_status'] == 'FLAGGED'
# Non-RFID rows will have NaN scan_status, so is_flagged will be False (coerced)
# If you want NaN instead of False for GPS/Temp rows, use:
# iot_cleaned_df['is_flagged'] = iot_cleaned_df['scan_status'].map({'FLAGGED': True, 'VERIFIED': False})

# ── temp_breach boolean ──────────────────────────────────────────────────────
# Needs goods_category per rfid_tag — call getShipment() on-chain for each tag
# GoodsCategory enum matches CATEGORY_MAP from the integration notebook

CATEGORY_INDEX_TO_NAME = {
    0: 'Deep Freeze',
    1: 'Frozen',
    2: 'Chill/Refrigerated',
    3: 'Pharma',
    4: 'Cool-Chain',
    5: 'Dry Goods',
    6: 'Electronics',
    7: 'Clothing',
    8: 'Industrial'
}

# Exact same ranges as smart-logistics-iot-simulation.ipynb — must match simulation
TEMP_RANGES = {
    'Deep Freeze':        (-30.0, -28.0),
    'Frozen':             (-20.0, -16.0),
    'Chill/Refrigerated': (2.0,    4.0),
    'Pharma':             (2.0,    8.0),
    'Cool-Chain':         (12.0,  14.0)
}

# Pull goods_category for each unique rfid_tag from the chain
tag_to_category    = {}
tag_to_origin      = {}
tag_to_destination = {}

all_tags = logistics_contract.functions.getAllRFIDTags().call()
for tag in all_tags:
    if tag == 'TEST-001':
        continue
    shipment = logistics_contract.functions.getShipment(tag).call()
    tag_to_category[tag]    = CATEGORY_INDEX_TO_NAME.get(shipment[1], None)
    tag_to_origin[tag]      = shipment[2]
    tag_to_destination[tag] = shipment[3]

# Add the three new columns
iot_cleaned_df['goods_category'] = iot_cleaned_df['rfid_tag'].map(tag_to_category)
iot_cleaned_df['origin']         = iot_cleaned_df['rfid_tag'].map(tag_to_origin)
iot_cleaned_df['destination']    = iot_cleaned_df['rfid_tag'].map(tag_to_destination)

# Map category onto temperature rows and compute breach
def compute_temp_breach(row):
    if row['sensor_type'] != 'Temperature':
        return None  # not applicable for GPS/RFID rows
    category = tag_to_category.get(row['rfid_tag'])
    if category not in TEMP_RANGES:
        return None  # non-temp-regulated goods category — no breach possible
    low, high = TEMP_RANGES[category]
    return not (low <= row['temperature_c'] <= high)

iot_cleaned_df['temp_breach'] = iot_cleaned_df.apply(compute_temp_breach, axis=1)

iot_cleaned_df = iot_cleaned_df[[
    'blockchain_timestamp',
    'sensor_timestamp',
    'rfid_tag',
    'device_id',
    'sensor_type',
    'latitude',
    'longitude',
    'scan_status',
    'temperature_c',
    'is_flagged',
    'temp_breach',
    'goods_category',
    'origin',
    'destination'
]]

print(f"Cleaned records: {len(iot_cleaned_df)}")
print(iot_cleaned_df.dtypes)
print(iot_cleaned_df.head())

Cleaned records: 329
blockchain_timestamp    datetime64[ns]
sensor_timestamp        datetime64[ns]
rfid_tag                        object
device_id                       object
sensor_type                     object
latitude                       float64
longitude                      float64
scan_status                     object
temperature_c                  float64
is_flagged                        bool
temp_breach                     object
goods_category                  object
origin                          object
destination                     object
dtype: object
  blockchain_timestamp    sensor_timestamp  rfid_tag device_id sensor_type  \
0  2026-06-04 16:21:45 2026-05-03 07:00:00  RFID-020    GPS291         GPS   
1  2026-06-04 16:21:46 2026-05-03 07:00:00  RFID-011    GPS728         GPS   
2  2026-06-04 16:21:48 2026-05-03 09:00:00  RFID-020    GPS920         GPS   
3  2026-06-04 16:21:48 2026-05-03 09:00:00  RFID-028    GPS356         GPS   
4  2026-06-04 16:21:51 2026-0

## 5. RFID Filter
Filters and groups records by rfid_tag for per-shipment analysis.

In [14]:
# ── RFID Filter ─────────────────────────────────────────────────────────────

iot_by_rfid = iot_cleaned_df.groupby('rfid_tag')
# groupby returns a GroupBy object — not a DataFrame
# use .get_group("RFID-001") in Stats to access one shipment's records

print(f"Unique shipments (RFID tags): {iot_by_rfid.ngroups}")
print("Groups:", list(iot_by_rfid.groups.keys()))

Unique shipments (RFID tags): 30
Groups: ['RFID-001', 'RFID-002', 'RFID-003', 'RFID-004', 'RFID-005', 'RFID-006', 'RFID-007', 'RFID-008', 'RFID-009', 'RFID-010', 'RFID-011', 'RFID-012', 'RFID-013', 'RFID-014', 'RFID-015', 'RFID-016', 'RFID-017', 'RFID-018', 'RFID-019', 'RFID-020', 'RFID-021', 'RFID-022', 'RFID-023', 'RFID-024', 'RFID-025', 'RFID-026', 'RFID-027', 'RFID-028', 'RFID-029', 'RFID-030']


## 6. Stats
NumPy descriptive statistics per sensor type — mean, min, max, std.

In [15]:
# ── Stats ───────────────────────────────────────────────────────────────────

stats_rows = []
for sensor_type, group in iot_cleaned_df.groupby('sensor_type'):
    if sensor_type == 'Temperature':
        vals = group['temperature_c'].dropna()
        col = 'temperature_c'
    elif sensor_type == 'GPS':
        vals = group['latitude'].dropna()
        col = 'latitude'
    else:
        continue  # RFID has no numeric column — skip

    if not vals.empty:
        stats_rows.append({
            'sensor_type': sensor_type,
            'column': col,
            'mean': np.mean(vals),
            'min': np.min(vals),
            'max': np.max(vals),
            'std': np.std(vals)
        })

sensor_stats_df = pd.DataFrame(stats_rows)
print(sensor_stats_df)

   sensor_type         column       mean        min        max        std
0          GPS       latitude  14.578048  14.264923  17.962386   0.395666
1  Temperature  temperature_c  -2.561053 -29.900000  16.400000  16.111769


## 7. Export
Pivot table construction and CSV export for downstream visualization.

In [16]:
# ── Export ──────────────────────────────────────────────────────────────────

# Full cleaned dataset — primary Tableau source
# Tidy format: one row per record, one column per variable
iot_cleaned_df.to_csv('../data/iot_cleaned_data.csv', index=False)
print("iot_cleaned_data.csv exported.")
print(f"Columns: {list(iot_cleaned_df.columns)}")

# NumPy stats summary — sensor-level summary for dashboard cards
# Already tidy from Stats cell — sensor_type, column, mean, min, max, std
sensor_stats_df.to_csv('../data/sensor_stats_summary.csv', index=False)
print("sensor_stats_summary.csv exported.")

iot_cleaned_data.csv exported.
Columns: ['blockchain_timestamp', 'sensor_timestamp', 'rfid_tag', 'device_id', 'sensor_type', 'latitude', 'longitude', 'scan_status', 'temperature_c', 'is_flagged', 'temp_breach', 'goods_category', 'origin', 'destination']
sensor_stats_summary.csv exported.


## 8. Preview
Sample output rows from each DataFrame.

In [17]:
# ── Preview ─────────────────────────────────────────────────────────────────

print("Pipeline complete. Sample records from iot_cleaned_df:")
display(iot_cleaned_df.head(10))

print(f"\nTotal clean records: {len(iot_cleaned_df)}")
print(f"Unique shipments: {iot_cleaned_df['rfid_tag'].nunique()}")
print(f"Sensor types: {iot_cleaned_df['sensor_type'].unique()}")

Pipeline complete. Sample records from iot_cleaned_df:


,blockchain_timestamp,sensor_timestamp,rfid_tag,device_id,sensor_type,latitude,longitude,scan_status,temperature_c,is_flagged,temp_breach,goods_category,origin,destination
0,2026-06-04 16:21:45,2026-05-03 07:00:00,RFID-020,GPS291,GPS,14.601956,120.989036,NaN,NaN,False,None,Industrial,Paco,Pandacan
1,2026-06-04 16:21:46,2026-05-03 07:00:00,RFID-011,GPS728,GPS,14.620032,120.962165,NaN,NaN,False,None,Cool-Chain,Sampaloc,Santa Ana
2,2026-06-04 16:21:48,2026-05-03 09:00:00,RFID-020,GPS920,GPS,14.595757,120.981906,NaN,NaN,False,None,Industrial,Paco,Pandacan
3,2026-06-04 16:21:48,2026-05-03 09:00:00,RFID-028,GPS356,GPS,14.374937,121.038127,NaN,NaN,False,None,Industrial,Bayanan,Marikina Heights
4,2026-06-04 16:21:51,2026-05-03 09:00:00,RFID-011,GPS262,GPS,14.622406,120.970944,NaN,NaN,False,None,Cool-Chain,Sampaloc,Santa Ana
5,2026-06-04 16:21:53,2026-05-03 11:00:00,RFID-020,GPS343,GPS,14.544307,120.958798,NaN,NaN,False,None,Industrial,Paco,Pandacan
6,2026-06-04 16:21:54,2026-05-03 11:00:00,RFID-028,GPS501,GPS,14.474628,121.042294,NaN,NaN,False,None,Industrial,Bayanan,Marikina Heights
7,2026-06-04 16:21:54,2026-05-03 11:00:00,RFID-011,GPS125,GPS,14.552821,120.960382,NaN,NaN,False,None,Cool-Chain,Sampaloc,Santa Ana
8,2026-06-04 16:21:56,2026-05-03 13:00:00,RFID-020,GPS858,GPS,14.550873,120.954035,NaN,NaN,False,None,Industrial,Paco,Pandacan
9,2026-06-04 16:21:58,2026-05-03 13:00:00,RFID-028,GPS805,GPS,14.521048,121.099622,NaN,NaN,False,None,Industrial,Bayanan,Marikina Heights



Total clean records: 329
Unique shipments: 30
Sensor types: ['GPS' 'Temperature' 'RFID']
